<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Package artifacts

**[C05-M08-L01 · Package artifacts](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/)** · C05, Machine Learning: From Problem to Reliable Model · Module 8, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** save the whole preprocessing-plus-model pipeline with its feature schema, threshold and versions, and prove in a fresh process that the saved model gives the same scores.

| | |
|---|---|
| **Time** | About 70 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Pipelines, from [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/). The threshold 0.19, from [Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/). |
| **You do not need** | The local project. The setup below downloads the data and the course's `ticket_model.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

In this module the **local project is the main route**, because a handoff is files and commands. This notebook does the same steps: it writes the files with `%%writefile` and runs them as commands with `!"{sys.executable}" train.py`. A command that starts with `!` runs in a new process, as in a terminal.

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M08-L01-package-artifacts/package-artifacts-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. `train.py` and `predict.py` import it. It is the final version, with 7 entries in `NUMBERS`.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. What must travel with the model

A recipe without the oven temperature gives a flat cake, even when every line of it is right. A model is the same: everything that it needs must travel with it. A file that training makes, and that another program uses later, is an **artifact**. The next cell fits a scaler on five order values (the tiny example from C04), saves it with `joblib.dump`, and loads it again with `joblib.load`.

> **Predict.** The mean is 54 and the SD is 26.53. What does the loaded scaler make of the value 400? Then run the cell.

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler

tiny_train = pd.DataFrame({"order_value": [20.0, 40.0, 50.0, 60.0, 100.0]})
scaler = StandardScaler().fit(tiny_train)
joblib.dump(scaler, "tiny_scaler.joblib")

again = joblib.load("tiny_scaler.joblib")
print("mean and SD:", again.mean_, again.scale_.round(2))
print("400 ->", again.transform(pd.DataFrame({"order_value": [400.0]})).round(2).ravel())

> **Check.** You should see `mean and SD: [54.] [26.53]` and `400 -> [13.04]`: (400 − 54) ÷ 26.53 = 13.04, the same as before saving.

Now the mistake: save only the model, and build a new scaler on each batch of new tickets. Today's batch has two orders, 400 and 380.

> **Predict.** What is the mean of 400 and 380? What will 400 become? Then run the cell.

In [ ]:
batch = pd.DataFrame({"order_value": [400.0, 380.0]})
rebuilt = StandardScaler().fit(batch)
print("mean and SD:", rebuilt.mean_, rebuilt.scale_)
print("400 ->", rebuilt.transform(batch.head(1)).round(2).ravel())

> **Check.** You should see `mean and SD: [390.] [10.]` and `400 -> [1.]`.

The same order gives 13.04 with the trained scaler and 1.0 with the rebuilt one. No error appears. This is Tomás's idea, "save only the model", and it is the forgotten oven temperature. The rule: **save everything that `fit` learned, in one object**: the pipeline.

## 2. The handoff contract

Priya's artifact is a folder `model/` with three files. The pipeline is the cake mix with every learned amount in it. `model_info.json` holds the facts that the receiver cannot guess, like the oven temperature. `check_sample.csv` is the photo of the finished cake, to compare with.

| File | What it holds |
|---|---|
| `escalation_model.joblib` | The fitted pipeline: preprocessing and model. |
| `model_info.json` | The **feature schema** (columns, kinds, known categories), the threshold, the versions, the checksums and the validation results. |
| `check_sample.csv` | 20 validation tickets with the scores that the model gave them at training time. |

A **schema** describes the data that a program expects: the column names, their types and the allowed values. The [diagram on the lesson page](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/#the-handoff-contract) shows how `train.py` writes the files and `predict.py` uses them.

## 3. Write train.py

The next four cells write `train.py`, in the four parts of the lesson. The first cell starts the file (`Writing train.py`); the others add to it (`Appending to train.py`). Run them in order.

Part 1: the imports, the review capacity, `choose_threshold()` (the rule from Module 6) and `sha256()` (a file's checksum).

In [ ]:
%%writefile train.py
"""Train the escalation model, choose its threshold, and save the artifact.

Nextia Learning, C05, Module 8, lesson 1 (Package artifacts).

    python train.py

Fits the pipeline on train only, chooses the threshold on valid for the
review capacity, and writes into model/:

    escalation_model.joblib   the fitted preprocessing-plus-model pipeline
    model_info.json           feature schema, threshold, versions, checksums, valid results
    check_sample.csv          20 valid tickets and their scores, to check a reload

It never reads the test set: evaluate.py does that, once.
"""

import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score

from ticket_model import CATEGORIES, DATA, FEATURES, NUMBERS, TARGET, build_pipeline, load

OUT = Path("model")
REVIEW_CAPACITY = 0.20  # the senior team can review at most 1 in 5 new tickets


def choose_threshold(scores, capacity):
    """The lowest threshold (in steps of 0.01) that flags no more than
    `capacity` of the tickets. Lower thresholds flag more tickets and catch
    more escalations, so the lowest one that fits catches the most."""
    for step in range(1, 100):
        threshold = step / 100
        if (scores >= threshold).mean() <= capacity:
            return threshold
    return 1.0


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

Part 2: fit on train only, score valid, choose the threshold, and compute the validation results.

In [ ]:
%%writefile -a train.py


def main():
    train, valid = load("train"), load("valid")
    pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])

    scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
    threshold = choose_threshold(scores, REVIEW_CAPACITY)
    flagged = scores >= threshold
    caught = int((flagged & (valid[TARGET] == 1)).sum())
    results = {
        "rows": len(valid), "escalated": int(valid[TARGET].sum()),
        "roc_auc": round(roc_auc_score(valid[TARGET], scores), 3),
        "average_precision": round(average_precision_score(valid[TARGET], scores), 3),
        "flagged": int(flagged.sum()), "flag_rate": round(flagged.mean(), 3),
        "precision": round(caught / flagged.sum(), 3),
        "recall": round(caught / valid[TARGET].sum(), 3),
    }

Part 3: save the **whole** pipeline with `joblib.dump`, and the first 20 validation tickets with their scores.

In [ ]:
%%writefile -a train.py

    OUT.mkdir(exist_ok=True)
    joblib.dump(pipeline, OUT / "escalation_model.joblib")
    sample = valid[["ticket_id", "created_at"] + FEATURES].head(20).copy()
    sample["score"] = scores[:20]
    sample.to_csv(OUT / "check_sample.csv", index=False)

Part 4: write `model_info.json`, with the schema taken from the fitted encoder, and print a summary.

In [ ]:
%%writefile -a train.py

    categories = pipeline.named_steps["prepare"].named_transformers_["categories"].categories_
    info = {
        "model": "Larkfield ticket escalation: logistic regression in a scikit-learn pipeline",
        "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        "target": f"{TARGET}: 1 if the ticket is escalated within 72 hours of creation",
        "prediction_time": "when the ticket is created",
        "features": {
            "categories": {name: [str(v) for v in values] for name, values in zip(CATEGORIES, categories)},
            "numbers": NUMBERS,
        },
        "threshold": threshold,
        "review_capacity": REVIEW_CAPACITY,
        "valid": results,
        "train_rows": len(train),
        "data_sha256": {name: sha256(DATA / name) for name in ("train.csv", "valid.csv", "extra_features.csv")},
        "artifact_sha256": sha256(OUT / "escalation_model.joblib"),
        "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                     "scikit-learn": sklearn.__version__, "numpy": np.__version__, "joblib": joblib.__version__},
    }
    (OUT / "model_info.json").write_text(json.dumps(info, indent=2) + "\n", encoding="utf-8")

    print(f"Trained on {len(train)} tickets. Threshold {threshold} for a review capacity of {REVIEW_CAPACITY:.0%}.")
    print("Valid: ROC AUC {roc_auc}, average precision {average_precision}, flagged {flagged} of {rows} "
          "({flag_rate:.1%}), precision {precision}, recall {recall}.".format(**results))
    print(f"Saved {OUT / 'escalation_model.joblib'}, model_info.json and check_sample.csv.")


if __name__ == "__main__":
    main()

The next cell checks that your `train.py` is the same as the course's file ([C05/project/train.py](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/project/train.py)). If it says "Not yet", run the four cells above again, in order.

In [ ]:
expect_hash('train.py', Path("train.py").read_text(encoding="utf-8"), '543aff4069db73c6')

Now run `train.py` as a command. `sys.executable` is the Python of this notebook, so the command uses the same packages.

> **Predict.** Which threshold will it choose? How many of the 1,179 validation tickets will it flag?

In [ ]:
import sys
!"{sys.executable}" train.py

> **Check.** You should see `Threshold 0.19`, `flagged 227 of 1179 (19.3%), precision 0.379, recall 0.47` and `Saved model/escalation_model.joblib, model_info.json and check_sample.csv.`

Look at the schema and the versions in `model_info.json`:

In [ ]:
import json
info = json.loads(Path("model/model_info.json").read_text(encoding="utf-8"))
for name, values in info["features"]["categories"].items():
    print(name, values)
print("numbers:", info["features"]["numbers"])
print("threshold:", info["threshold"])
print("versions:", info["versions"])

> **Check.** You should see 5 channels, 5 teams, 4 segments and 5 regions, 7 numbers, and `threshold: 0.19`. The versions are the ones of this notebook. Locally the course has Python 3.14.6, numpy 2.5.3 and joblib 1.6.0. Colab and Kaggle have their own Python and numpy versions, so you may see other numbers there.

> **Your turn.** The model inside the pipeline does not see the 11 ticket columns. It sees the columns that the preprocessing makes. Load the saved pipeline with `joblib.load`. Then transform the first validation ticket with its `"prepare"` step, and set `n_columns` to the number of columns. (Use `load("valid")` and `FEATURES` from `ticket_model`.)

In [ ]:
from ticket_model import FEATURES, load

pipeline = joblib.load("model/escalation_model.joblib")
n_columns = pipeline.named_steps["prepare"].transform(load("valid")[FEATURES].head(1)).shape[1]
print(n_columns)

In [ ]:
expect_hash('n_columns', n_columns, '59e19706d51d39f6')

The categories become one column each (one-hot encoding), and each number with gaps in train gets an extra 0/1 column. Only the fitted preprocessing knows exactly which columns, in which order. Section 8 shows what happens without it.

## 4. Pin every version: requirements.txt

The artifact needs the same package versions to load the same way: also numpy (the arrays inside the pipeline) and joblib (the file format). Run the cell to see the versions in this notebook:

In [ ]:
import numpy, sklearn
print("numpy", numpy.__version__, "| joblib", joblib.__version__, "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

> **Check.** You should see `scikit-learn 1.9.1` and `pandas 3.0.6`. Locally you see `numpy 2.5.3 | joblib 1.6.0`.

The next cell writes the course's `requirements.txt`: five lines, each with an exact version (`==`). In your local project, remove the `ipykernel` line: it is for your editor, not for the model.

In [ ]:
%%writefile requirements.txt
pandas==3.0.6
scikit-learn==1.9.1
numpy==2.5.3
joblib==1.6.0
matplotlib==3.11.2

## 5. A fresh process: predict.py --check

A check in this notebook proves little: the notebook still has the pipeline and your variables in memory. A **fresh process** starts with nothing in memory, like the person who receives the files. A `!` command starts a new process each time, so it is a fresh process.

The next cell writes the first version of `predict.py`. Lesson 3 adds the input checks.

In [ ]:
%%writefile predict.py
"""Score new tickets with the saved escalation model.

Nextia Learning, C05, Module 8 (Package artifacts; Plan production checks).

    python predict.py data/july_tickets.csv predictions.csv
    python predict.py --check     reload the model and compare the fixed sample
"""

import argparse
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from ticket_model import FEATURES, read

MODEL = Path("model")


def load_model():
    info = json.loads((MODEL / "model_info.json").read_text(encoding="utf-8"))
    return joblib.load(MODEL / "escalation_model.joblib"), info


def check_sample(pipeline):
    sample = read(MODEL / "check_sample.csv")
    scores = pipeline.predict_proba(sample[FEATURES])[:, 1]
    largest = float(np.abs(scores - sample["score"]).max())
    if largest > 1e-9:
        sys.exit(f"Check failed: the reloaded model's scores differ from the saved ones by up to {largest:.2e}.")
    print(f"Check passed: {len(sample)} scores are identical after reloading (largest difference {largest:.1e}).")


def main():
    parser = argparse.ArgumentParser(description="Score new tickets with the saved escalation model.")
    parser.add_argument("input", nargs="?", help="CSV file with one row per new ticket")
    parser.add_argument("output", nargs="?", default="predictions.csv")
    parser.add_argument("--check", action="store_true", help="compare the fixed sample after reloading")
    args = parser.parse_args()
    pipeline, info = load_model()
    if args.check:
        check_sample(pipeline)
        return
    if not args.input:
        parser.error("give an input CSV file, or --check")

    df = read(args.input)
    scores = pipeline.predict_proba(df[FEATURES])[:, 1]
    out = pd.DataFrame({"ticket_id": df["ticket_id"], "score": scores.round(4),
                        "flag": (scores >= info["threshold"]).astype(int)})
    out.to_csv(args.output, index=False)
    print(f"Scored {len(out)} tickets with threshold {info['threshold']}: {out['flag'].sum()} flagged "
          f"({out['flag'].mean():.1%}). Wrote {args.output}.")


if __name__ == "__main__":
    main()

> **Predict.** The saved scores went through a CSV file. Will the largest difference be exactly 0? Then run the check.

In [ ]:
!"{sys.executable}" predict.py --check

> **Check.** You should see `Check passed: 20 scores are identical after reloading (largest difference 1.1e-16).` Reading a number back from text can change its last digit. 1.1e-16 is far below the limit of 1e-9.

> **Your turn.** Do the same check here, by hand. Load the artifact, read `model/check_sample.csv` with `read()` from `ticket_model`, score the 20 tickets, and set `largest` to the largest absolute difference from the column `score`.

In [ ]:
from ticket_model import read

sample = read("model/check_sample.csv")
reloaded = joblib.load("model/escalation_model.joblib")
largest = float(np.abs(reloaded.predict_proba(sample[FEATURES])[:, 1] - sample["score"]).max())
print(largest)

In [ ]:
if largest is Ellipsis or largest is None:
    print("Not yet: largest has no value. Write your answer in the cell above, then run it again.")
elif not isinstance(largest, (float, int, np.floating)):
    print("Not yet: largest must be one number, for example from .max().")
elif largest < 1e-9:
    print("Check passed: the reloaded model gives the same scores.")
else:
    print("Not yet: the difference is too large. Did you use predict_proba(...)[:, 1] and the column score?")

Now score the July tickets, which arrive after the training:

In [ ]:
!"{sys.executable}" predict.py data/july_tickets.csv predictions.csv

> **Check.** You should see `Scored 1400 tickets with threshold 0.19: 282 flagged (20.1%). Wrote predictions.csv.` It works, and it says nothing more. [Plan production checks](https://learning.nextia-ai.com/courses/ml/m08/plan-production-checks/) shows what it does not tell you.

## 6. Load only files you trust

> **Warning.** `joblib.load` (and `pickle`) runs instructions that are stored in the file. Load only artifacts that you made or got from a source you trust.

The next cell makes a harmless file that calls `print` when it is loaded. Run it.

In [ ]:
class Surprise:
    def __reduce__(self):
        return (print, ("This line ran while the file was loading.",))

joblib.dump(Surprise(), "surprise.joblib")
joblib.load("surprise.joblib")

> **Check.** You should see `This line ran while the file was loading.` A malicious file could run any code in the same way.

## 7. When versions differ

Suppose the file was made with scikit-learn 1.8.0 and you have 1.9.1. To show it without installing an old version, the next cell changes the version text inside a **copy** of the artifact, then loads the copy.

In [ ]:
import warnings

raw = Path("model/escalation_model.joblib").read_bytes()
Path("old_version.joblib").write_bytes(raw.replace(b"1.9.1", b"1.8.0"))
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    joblib.load("old_version.joblib")
print(len(caught), "warnings. The first one:")
print(type(caught[0].message).__name__ + ":", caught[0].message)

> **Check.** You should see `12 warnings` and `InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.9.1.` One warning comes for each estimator inside the pipeline.

The file loads, but scikit-learn does not promise that it works the same. Do not ignore the warning: install the pinned versions, or train again and make a new artifact.

## 8. Your turn: a smaller review team

Grace's senior team can review only 15% of new tickets in August.

> **Predict.** Will the threshold go up or down? Precision? Recall? Will `predict.py --check` still pass?

> **Your turn.** The cell changes `REVIEW_CAPACITY` to 0.15 and runs `train.py`. Complete it: set `new_threshold` to the threshold in the new `model/model_info.json`.

In [ ]:
text = Path("train.py").read_text(encoding="utf-8")
Path("train.py").write_text(text.replace("REVIEW_CAPACITY = 0.20", "REVIEW_CAPACITY = 0.15"), encoding="utf-8")
!"{sys.executable}" train.py
!"{sys.executable}" predict.py --check

new_threshold = json.loads(Path("model/model_info.json").read_text(encoding="utf-8"))["threshold"]
print(new_threshold)

In [ ]:
expect_hash('new_threshold', new_threshold, 'dc08a4e8bbdd4019')

> **Check.** You should see `flagged 174 of 1179 (14.8%), precision 0.431, recall 0.41`, and `Check passed`. The threshold goes up: fewer tickets may be flagged, so precision goes up and recall goes down. ROC AUC and average precision do not change: they do not depend on the threshold.

Run the next cell to set the capacity back to 20% and make the course's artifact again. You should see `Threshold 0.19`.

In [ ]:
text = Path("train.py").read_text(encoding="utf-8")
Path("train.py").write_text(text.replace("REVIEW_CAPACITY = 0.15", "REVIEW_CAPACITY = 0.20"), encoding="utf-8")
!"{sys.executable}" train.py

## 9. Failure case: save only the classifier

Tomás saves only the model, `pipeline.named_steps["model"]`, and prepares the data again in the scoring program. The next cell tries three ways. Each error is caught and printed.

> **Predict.** The model expects 28 columns. Which tries will fail, and how?

In [ ]:
from ticket_model import build_pipeline

saved = joblib.load("model/escalation_model.joblib")
model_only = saved.named_steps["model"]
sample = read("model/check_sample.csv")
july = read("data/july_tickets.csv")
try:
    model_only.predict_proba(sample[FEATURES])           # 1. the raw ticket columns
except ValueError as error:
    print("1.", error)
for name, rows in (("2. 20 check tickets:", sample), ("3. July tickets:", july)):
    prepare = build_pipeline().named_steps["prepare"].fit(rows[FEATURES])   # preprocessing fitted again
    try:
        model_only.predict_proba(prepare.transform(rows[FEATURES]))
    except ValueError as error:
        print(name, error)

> **Check.** You should see `could not convert string to float: 'chat'`, then `X has 22 features` for the 20 tickets (not every category is in them), then `X has 29 features` for July (a new channel, `social`). A `UserWarning` ("X has feature names, but LogisticRegression was fitted without feature names") may appear too: one more sign that the model alone never saw ticket columns.

The errors are the lucky case. On the validation tickets, every category is present, so the same mistake runs with no error:

In [ ]:
valid = load("valid")
prepare = build_pipeline().named_steps["prepare"].fit(valid[FEATURES])     # fitted on May, not on train
wrong = model_only.predict_proba(prepare.transform(valid[FEATURES]))[:, 1]
right = saved.predict_proba(valid[FEATURES])[:, 1]
print("flagged:", (wrong >= 0.19).sum(), "instead of", (right >= 0.19).sum())
print("tickets that change their flag:", ((wrong >= 0.19) != (right >= 0.19)).sum())
print("largest score difference:", round(float(np.abs(wrong - right).max()), 3))

> **Check.** You should see `flagged: 234 instead of 227`, `7` tickets that change their flag, and a largest difference of `0.025`. The medians, means and SDs came from May, not from train. The fix: save the whole pipeline, and never fit anything in the scoring program.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

You wrote `train.py`, which saves the whole pipeline, its schema, threshold, versions and checksums, and a check sample in `model/`. You pinned five packages in `requirements.txt`. `predict.py --check` proved in a fresh process that the loaded model gives the same scores. Load only artifacts that you trust, and fix an `InconsistentVersionWarning` instead of ignoring it.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Document use boundaries](https://learning.nextia-ai.com/courses/ml/m08/document-use-boundaries/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.